# Your first PINN, in fifteen lines

**Book:** Chapter 2, Listing 2.1 and Figure 2.1 &nbsp;·&nbsp; `ch02/decay_pinn_15lines.ipynb`

Governing equation

$$\frac{du}{dt} = -k\,u,\qquad u(0)=1,\qquad k=2,\qquad t\in[0,3],$$

exact solution $u(t)=e^{-kt}$ (used **only** to verify — never to train).

The initial condition is imposed **exactly** by the trial function $u(t)=1+t\,\mathcal N(t)$, so

$$\mathcal{L}=\frac{1}{N_f}\sum_{i=1}^{N_f}\Big(\frac{du}{dt}\Big|_{t_i}+k\,u(t_i)\Big)^2$$

is the *whole* loss: one term, no boundary penalty, no weight to choose.

In [ ]:
import time
import torch, torch.nn as nn
from torch.autograd import grad
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)

# ---------------- the whole PINN (15 lines) ----------------
k = 2.0                                          # decay rate
net = nn.Sequential(nn.Linear(1,32), nn.Tanh(),
                    nn.Linear(32,32), nn.Tanh(), nn.Linear(32,1))
u_of = lambda t: 1 + t*net(t)                    # trial fn: u(0)=1 holds EXACTLY
opt = torch.optim.Adam(net.parameters(), lr=3e-3)

hist = []
t0 = time.perf_counter()
for step in range(4000):
    opt.zero_grad()
    t = (torch.rand(256,1)*3).requires_grad_(True)          # collocation in [0,3]
    u = u_of(t)
    u_t = grad(u, t, torch.ones_like(u), create_graph=True)[0]
    res = u_t + k*u                                          # the ODE residual
    loss = (res**2).mean()                                   # loss = residual only
    loss.backward()                                          # nothing to weight
    opt.step()
    hist.append(loss.item())
# -----------------------------------------------------------
train_time = time.perf_counter() - t0

tg = torch.linspace(0, 3, 400).reshape(-1,1)
with torch.no_grad(): up = u_of(tg).numpy().ravel()
te = np.exp(-k*tg.numpy().ravel())
err = np.sqrt(np.mean((up-te)**2)/np.mean(te**2))
print(f'training: {train_time:.1f} s')
print(f'relative L2 error: {err:.1e}')

fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.2))
ax[0].plot(tg.numpy(), te, 'g', lw=2.6, alpha=.6, label='exact $e^{-kt}$')
ax[0].plot(tg.numpy(), up, 'r--', lw=1.6, label=f'PINN (rel $L_2$={err:.1e})')
ax[0].set_xlabel('t'); ax[0].set_ylabel('u'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[0].set_title('(a) Solved from the equation alone — no data')
ax[1].semilogy(hist, lw=.8)
ax[1].set_xlabel('Adam step'); ax[1].set_ylabel('loss  (mean squared residual)')
ax[1].grid(alpha=.3); ax[1].set_title('(b) The loss — a single term')
plt.tight_layout(); plt.show()